In [1]:
import torch

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math
import torch.nn.functional as F

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
def cal_part_logprob_in_batch(_input_text: list, _tokenizer, _device, _max_length=512):
    with torch.no_grad():
        inputs = _tokenizer(_input_text, return_tensors="pt", padding=True, truncation=True, max_length=_max_length).to(_device)
        outputs = model(**inputs, labels=inputs["input_ids"], loss_type='ForCausalLMLoss')
    
        logits = outputs.logits.to('cpu')
        del outputs
        inputs = inputs.to('cpu')
        attention_mask = inputs["attention_mask"]
        input_ids = inputs["input_ids"]
        print(input_ids.shape)
        
        shift_logits = logits[: , :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        
        log_probs = F.log_softmax(shift_logits, dim=-1)
        
        log_probs_for_tokens = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
        actual_log_probs_for_tokens = attention_mask[:, 1:] * log_probs_for_tokens
        
        actual_nums = attention_mask[:, 1:].sum(dim=1)
        actual_sums = actual_log_probs_for_tokens.sum(dim=1)
        avg_logProbs = actual_sums/actual_nums
    return avg_logProbs.tolist()

##### Idea: calcuate the average logprob per token in a sentence, passage, or a multi-passage passage, it seems not difficult to calculate

In [4]:
import sys
import os
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
analysis_root = os.path.abspath(os.path.join(notebook_dir, '..', 'analysis'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if analysis_root not in sys.path:
    sys.path.insert(0, analysis_root)

from analysis.tools import coherence_cal

In [5]:
torch.cuda.empty_cache()

In [6]:
_ret = 'bm25'
res, doc_dict, doc_length_dict = coherence_cal.get_res_and_dicts('nq_test', _ret)

In [12]:
import json
from tqdm import tqdm

_k = 12
output_path = f'./log_prob_temp_res/nq_test_{_ret}_{_k}.json'

try:
    f = open(output_path, 'r')
    prob_res = json.load(f)
    f.close()
except:
    prob_res = {}

for qid in tqdm(res.qid.unique()):
    if(qid in prob_res.keys()):
        continue
    _res_per_q = {}

    _batch_size = min(_k, 20)
    _i = 0
    while _i < _k:
        doc_texts = res[(res.qid==qid)&(res['rank']>=_i)&(res['rank']<_i+_batch_size)].docno.apply(lambda x: doc_dict[str(x)]).tolist()
        _res_per_q = {}
        _pid = 0
        avg_logProb_list = cal_part_logprob_in_batch(doc_texts, tokenizer, device)
        torch.cuda.empty_cache()
        _res_per_q.update(dict(zip(range(_i, _i+len(doc_texts)), avg_logProb_list)))
        _i += _batch_size

    prob_res.update({qid: _res_per_q})

    f = open(output_path, 'w')
    json.dump(prob_res, f)
    f.close()

100%|██████████| 3610/3610 [56:43<00:00,  1.06it/s] 


In [8]:
import json
from tqdm import tqdm
import math

_k = 10
prob_res = {}
output_path = f'./log_prob_temp_res/full_context/nq_test_{_ret}_{_k}.json'

try:
    f = open(output_path, 'r')
    prob_res = json.load(f)
    f.close()
except:
    prob_res = {}

_max_tokens = 512 + 200*max(0, _k-5)
_batch_size = math.floor(3200/_max_tokens)
_i = 0
doc_texts = []
_qid_to_write = []
for qid in tqdm(res.qid.unique()):
    if(qid in prob_res.keys()):
        continue
    doc_text = ''.join(res[(res.qid==qid)&(res['rank']<_k)].docno.apply(lambda x: doc_dict[str(x)]).tolist())
    doc_texts.append(doc_text)
    _qid_to_write.append(qid)
    _i += 1
    
    if ((_i == _batch_size)|(qid == res.qid.unique()[-1])):
        avg_logProb_list = cal_part_logprob_in_batch(doc_texts, tokenizer, device, _max_tokens)
        torch.cuda.empty_cache()
        prob_res.update(dict(zip(_qid_to_write, avg_logProb_list)))
        _qid_to_write = []
        doc_texts = []
        _i = 0
        
        f = open(output_path, 'w')
        json.dump(prob_res, f)
        f.close()



  0%|          | 0/3610 [00:00<?, ?it/s]

torch.Size([2, 1324])


  8%|▊         | 300/3610 [00:02<00:27, 120.70it/s]

torch.Size([2, 1275])
torch.Size([2, 1336])
torch.Size([2, 1352])
torch.Size([2, 1357])
torch.Size([2, 1362])
torch.Size([2, 1388])


  9%|▊         | 313/3610 [00:12<02:53, 19.02it/s] 

...

torch.Size([2, 1396])


 12%|█▏        | 418/3610 [01:40<43:47,  1.21it/s]

torch.Size([2, 1329])


 12%|█▏        | 421/3610 [01:43<13:07,  4.05it/s]


KeyboardInterrupt: 